# Preparing an Alpaca-Style Instruction Dataset from TinyStories

TinyStories has no natural instructions — just plain short stories. To use it
for instruction fine-tuning, we need to **synthesize** `{instruction, input,
output}` triples from the raw text.

This notebook builds that pipeline step by step, so every heuristic is visible
and testable in isolation before we run it over the full dataset. The
productionized, CLI-driven version of everything below lives in
`training/gpt/build_alpaca_dataset.py` — this notebook is where that script
was designed and validated.

**Steps:**
1. Setup & imports
2. Load a small slice of TinyStories to explore
3. Decide length filters (skip stories that are too short/long)
4. Build and test each heuristic (keyword extraction, sentence splitting, story continuation)
5. Build the four instruction templates and test each on a sample story
6. Run the full pipeline over a larger subset
7. Inspect the resulting dataset
8. Save to JSONL
9. Run the pipeline over the full train/validation splits


## 1. Setup & imports

In [1]:
import re
import json
import random
from collections import Counter

from datasets import load_dataset
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

SEED = 1337
rng = random.Random(SEED)


## 2. Load a small slice of TinyStories to explore

We only pull a handful of stories here — enough to design and sanity-check
the heuristics against real examples, without waiting on the full dataset
download/tokenize cycle every time we tweak something.

In [2]:
dataset = load_dataset("roneneldan/TinyStories", cache_dir="tiny_stories")
sample_stories = dataset["train"]["text"][:100]

print(f"Pulled {len(sample_stories)} sample stories.\n")
print(sample_stories[0])


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Pulled 100 sample stories.

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


## 3. Decide length filters

Very short stories don't give templates like `continuation` enough text to
split meaningfully, and very long ones risk blowing past the model's
`context_length` (1024) once we add an instruction on top. We filter both
out rather than truncating awkwardly.

In [3]:
MIN_WORDS = 40
MAX_WORDS = 600

word_counts = [len(s.split()) for s in sample_stories]
print("Word count — min:", min(word_counts), "max:", max(word_counts),
      "mean:", round(sum(word_counts) / len(word_counts), 1))

kept = [s for s in sample_stories if MIN_WORDS <= len(s.split()) <= MAX_WORDS]
print(f"{len(kept)}/{len(sample_stories)} sample stories pass the length filter")


Word count — min: 80 max: 212 mean: 147.7
100/100 sample stories pass the length filter


## 4. Build and test each heuristic in isolation

These are three small, reusable building blocks. On their own they don't
produce instruction data — they just extract or split pieces of text. In
Section 5, each one gets wired into a specific instruction template to
actually build `{instruction, input, output}` examples. We test each helper
here on its own first, so if a template produces something odd later, we'll
already know whether the bug lives in the helper or in how the template uses
it.

- **`extract_keywords`** — pulls a few content words out of a story, later
  used to build the *"write a story that includes these words"* template.
  Example: given a story about Lily and Max playing fetch in the park, it
  might return `['park', 'ball', 'happily']` — words specific enough to
  meaningfully constrain a new story, with common filler words ("the", "and",
  "was") excluded.

- **`extract_first_sentence`** — splits a story into its opening sentence and
  everything after it, later used to build the *"continue from this opening
  sentence"* template.
  Example: `"Once upon a time, there was a little girl named Lily. She loved
  to play in the park."` splits into first sentence `"Once upon a time, there
  was a little girl named Lily."` and remainder `"She loved to play in the
  park."`

- **`split_continue`** — splits a story into a prefix and the remainder at a
  random cut point (a fraction of the way through, not necessarily a sentence
  boundary), later used to build the *"continue this partial story"*
  template.
  Example: with `frac=0.3` on a 100-word story, the first ~30 words become
  the prefix (`input`) and the remaining ~70 words become what the model must
  generate (`output`).

In [4]:
# scikit-learn's built-in English stopword list
STOPWORDS = set(ENGLISH_STOP_WORDS)

def extract_keywords(text, n=3, rng=random):
    words = re.findall(r"[A-Za-z]+", text)
    candidates = [w.lower() for w in words if w.lower() not in STOPWORDS and len(w) > 3]
    unique = list(dict.fromkeys(candidates))  # de-dup, preserve first-seen order
    if len(unique) <= n:
        return unique
    return rng.sample(unique, n)

# Test on a real sample story
test_story = kept[0]
print(test_story[:200], "...\n")
print("Extracted keywords:", extract_keywords(test_story, n=3, rng=rng))


One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on ...

Extracted keywords: ['sharing', 'shared', 'thanked']


In [5]:
def extract_first_sentence(text):
    match = re.match(r"\s*(.+?[.!?])(\s+|$)", text.strip())
    if not match:
        return text.strip(), ""
    first = match.group(1).strip()
    rest = text.strip()[match.end():].strip()
    return first, rest

first, rest = extract_first_sentence(test_story)
print("First sentence:", first)
print("Remainder starts with:", rest[:80], "...")


First sentence: One day, a little girl named Lily found a needle in her room.
Remainder starts with: She knew it was difficult to play with it because it was sharp. Lily wanted to s ...


In [6]:
def split_continue(text, frac):
    words = text.strip().split()
    cut = max(1, int(len(words) * frac))
    cut = min(cut, len(words) - 1)  # always leave something to generate
    return " ".join(words[:cut]), " ".join(words[cut:])

prefix, remainder = split_continue(test_story, frac=0.3)
print("Prefix (30%):", prefix)
print()
print("Remainder:", remainder[:120], "...")


Prefix (30%): One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a

Remainder: button on her shirt. Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt ...


## 5. Build the four instruction templates

Each template turns one story into one `{instruction, input, output}`
example. Note where the variable content lives differs by template — some
put it in `input` (the classic Alpaca pattern), others fold it directly into
`instruction` because the constraint *is* the instruction (e.g. "write using
these words" doesn't split cleanly into task + content).

**1. `continuation`** — "Here's part of a story, keep going."
- `instruction` (fixed): *"Continue the following children's story in a
  natural and coherent way."*
- `input`: the first 20–40% of the story (random cut point, via
  `split_continue`)
- `output`: everything after that cut point
- Teaches the model to pick up narrative threads mid-story — tracking
  characters, tone, and plot direction without being told them explicitly.

**2. `opening_sentence`** — "Here's one sentence, write the full story."
- `instruction` (fixed): *"Write a short children's story that begins with
  the given sentence."*
- `input`: just the first sentence (via `extract_first_sentence`)
- `output`: the rest of the story
- Similar skill to `continuation`, but from a much smaller seed — forces the
  model to generate more from less.

**3. `keyword_constrained`** — "Write a story that includes these words."
- `instruction` (variable): *"Write a short children's story that includes
  the following words: X, Y, Z."* — the word list is baked directly into the
  instruction text
- `input`: empty
- `output`: the full story
- A genuinely different skill from the first two — constrained generation
  from scratch, closest to what real datasets like `TinyStoriesInstruct` do
  with their `Words:` field.

**4. `free_generation`** — "Just write me a story."
- `instruction` (fixed): *"Write a short, simple story suitable for a young
  child."*
- `input`: empty
- `output`: the full story
- The simplest, most generic instruction — no constraints, no seed text.
  Teaches the model that `input` isn't mandatory, and gives a baseline "just
  write something reasonable" behavior.

**Why all four together:** each story is randomly assigned 2 of these 4
templates (`examples_per_story=2`), so across the dataset the model sees the
same underlying story-writing capability invoked through structurally
different instructions — continuing partial text, generating from a single
sentence, generating from a word list, or generating from nothing at all.
That variety is the actual point of instruction-tuning: mapping many
different phrasings and input shapes onto "produce an appropriate children's
story," rather than memorizing one fixed instruction format.

We test each template individually on the same sample story before combining
them, so we can eyeball that each produces sensible output.

In [7]:
def make_continuation_example(story, rng):
    frac = rng.uniform(0.2, 0.4)
    prefix, rest = split_continue(story, frac)
    if not prefix or not rest:
        return None
    return {
        "instruction": "Continue the following children's story in a natural and coherent way.",
        "input": prefix,
        "output": rest,
    }

def make_opening_sentence_example(story, rng):
    first, rest = extract_first_sentence(story)
    if not first or not rest:
        return None
    return {
        "instruction": "Write a short children's story that begins with the given sentence.",
        "input": first,
        "output": rest,
    }

def make_keyword_example(story, rng):
    n = rng.choice([2, 3, 4])
    kws = extract_keywords(story, n=n, rng=rng)
    if len(kws) < 2:
        return None
    return {
        "instruction": f"Write a short children's story that includes the following words: {', '.join(kws)}.",
        "input": "",
        "output": story.strip(),
    }

def make_free_generation_example(story, rng):
    return {
        "instruction": "Write a short, simple story suitable for a young child.",
        "input": "",
        "output": story.strip(),
    }

TEMPLATES = [
    ("continuation", make_continuation_example),
    ("opening_sentence", make_opening_sentence_example),
    ("keyword_constrained", make_keyword_example),
    ("free_generation", make_free_generation_example),
]


In [8]:
# Sanity-check every template against the same story
for name, builder in TEMPLATES:
    ex = builder(test_story, rng)
    print(f"--- {name} ---")
    print("instruction:", ex["instruction"])
    print("input:      ", ex["input"][:100])
    print("output:     ", ex["output"][:100], "...")
    print()


--- continuation ---
instruction: Continue the following children's story in a natural and coherent way.
input:       One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with
output:      she could sew a button on her shirt. Lily went to her mom and said, "Mom, I found this needle. Can y ...

--- opening_sentence ---
instruction: Write a short children's story that begins with the given sentence.
input:       One day, a little girl named Lily found a needle in her room.
output:      She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with ...

--- keyword_constrained ---
instruction: Write a short children's story that includes the following words: fixing, room, felt, worked.
input:       
output:      One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with ...

--- free_generation ---
instruction: Write a short, simple story suitable for a young child.
i

## 6. Run the full pipeline over a larger subset

Now we combine everything: filter by length, randomly pick `examples_per_story`
distinct templates per story, and build the dataset. Using 2 templates per
story (rather than all 4) keeps the same story's content from dominating the
dataset with too many near-duplicate examples.

In [9]:
def build_examples_for_story(story, examples_per_story, rng):
    story = story.strip()
    word_count = len(story.split())
    if word_count < MIN_WORDS or word_count > MAX_WORDS:
        return []

    chosen = rng.sample(TEMPLATES, k=min(examples_per_story, len(TEMPLATES)))
    examples = []
    for name, builder in chosen:
        ex = builder(story, rng)
        if ex is not None:
            ex["template"] = name  # for inspection only — drop before training
            examples.append(ex)
    return examples

def build_dataset(stories, examples_per_story=2, seed=SEED):
    build_rng = random.Random(seed)
    dataset_out = []
    skipped = 0
    for story in stories:
        examples = build_examples_for_story(story, examples_per_story, build_rng)
        if not examples:
            skipped += 1
        dataset_out.extend(examples)
    print(f"Built {len(dataset_out)} instruction examples from {len(stories)} stories "
          f"({skipped} stories skipped as too short/long).")
    return dataset_out


In [10]:
# Pull a larger slice for a more representative run. Start small (a few
# thousand) here — the full-dataset run belongs in build_alpaca_dataset.py
# via the CLI, not in an interactive notebook.
num_stories = 2000
stories_subset = dataset["train"]["text"][:num_stories]

alpaca_examples = build_dataset(stories_subset, examples_per_story=2, seed=SEED)
random.Random(SEED).shuffle(alpaca_examples)


Built 3991 instruction examples from 2000 stories (4 stories skipped as too short/long).


## 7. Inspect the resulting dataset

In [11]:
counts = Counter(ex["template"] for ex in alpaca_examples)
print("Template distribution:")
for name, count in counts.most_common():
    print(f"  {name}: {count}")


Template distribution:
  free_generation: 1020
  keyword_constrained: 1017
  continuation: 990
  opening_sentence: 964


In [12]:
# Look at a few random examples end to end
for ex in rng.sample(alpaca_examples, 3):
    print(f"[{ex['template']}]")
    print("INSTRUCTION:", ex["instruction"])
    print("INPUT:      ", ex["input"][:150])
    print("OUTPUT:     ", ex["output"][:150], "...")
    print("-" * 60)


[opening_sentence]
INSTRUCTION: Write a short children's story that begins with the given sentence.
INPUT:       Once upon a time, in a big forest, there was a rhinoceros named Rosie.
OUTPUT:      Rosie loved to bounce up and down all day long. She would bounce on the soft grass and make her friends giggle.

One day, Rosie met a little bird who  ...
------------------------------------------------------------
[keyword_constrained]
INSTRUCTION: Write a short children's story that includes the following words: said, walk, want, sure.
INPUT:       
OUTPUT:      Mommy and Jimmy were walking down a path one night. It was dark, and their shadows were long. 
Jimmy said, "Mommy, look at my shadow. I want to let it ...
------------------------------------------------------------
[keyword_constrained]
INSTRUCTION: Write a short children's story that includes the following words: watch, journey, goodbye.
INPUT:       
OUTPUT:      Once upon a time there were two friends named Fluffy and Daisy. Th

## 8. Save to JSONL

Drop the `template` field before saving — it was only there for our own
inspection above; the model should never see it as part of the actual
instruction/response pair.

In [13]:
def save_jsonl(examples, path):
    with open(path, "w", encoding="utf-8") as f:
        for ex in examples:
            ex_clean = {k: v for k, v in ex.items() if k != "template"}
            f.write(json.dumps(ex_clean, ensure_ascii=False) + "\n")
    print(f"Saved {len(examples)} examples to {path}")

save_jsonl(alpaca_examples, "tinystories_alpaca_train_sample.jsonl")


Saved 3991 examples to tinystories_alpaca_train_sample.jsonl


### 9. Run the pipeline over the full train/validation splits

9.1 Download build_alpaca_dataset.py from GitHub to your computer if you haven't already, so it's ready to select in the upload dialog.

9.2 Run the upload cell

In [14]:
from google.colab import files
uploaded = files.upload()

Saving build_alpaca_dataset.py to build_alpaca_dataset.py


9.3 Run it

In [15]:
!python build_alpaca_dataset.py \
    --split train \
    --num_stories 1000000 \
    --output_dir ./sft_data

Built 1993466 instruction examples from 1000000 stories (3230 stories skipped as too short/long).
Template distribution:
  continuation: 498858
  opening_sentence: 498739
  free_generation: 498194
  keyword_constrained: 497675
Saved 1993466 examples to ./sft_data/tinystories_alpaca_train.jsonl


9.4 Verify the output

In [16]:
!head -n 3 sft_data/tinystories_alpaca_train.jsonl

{"instruction": "Continue the following children's story in a natural and coherent way.", "input": "Lily and Ben were best friends. They liked to play with crayons and paper. One day, they decided to perform a show for their moms. They drew pictures of animals and people with their crayons. They also found a bottle of ink in the closet. \"Let's use this ink to make our pictures more pretty,\" Ben said. He opened the bottle and poured some ink on his paper. He made a big black spot. Lily wanted to try too. She took the bottle and poured some ink on her paper. But she poured too much. The ink spilled on the floor and on", "output": "her dress. She made a big mess. \"Oh no!\" Lily cried. \"Mom will be so mad at me. Look at my dress. It is all black and wet.\" Ben felt sorry for Lily. He tried to help her clean the ink. But it was too hard. The ink did not come off. It stained everything. Their moms came to see their show. They saw the ink on the floor and on Lily's dress. They were not ha